# Seestar S30 Pro — Variable Star Photometry Analysis

Interactive analysis notebook for data produced by the variable star pipeline.  
Plots are publication-quality (300 dpi PNG + PDF vector).

**Data source**: `results/<STAR_NAME>/photometry.csv`  
**Produced by**: `seestar_varstar_siril.py` + Siril 1.4

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# Import functions from the plotting module
sys.path.insert(0, str(Path(".").resolve()))
from plot_lightcurve import (
    load_photometry, sigma_clip, weighted_bin, phase_fold, lomb_scargle,
    plot_light_curve, plot_diagnostics, plot_phase_folded, plot_periodogram,
    RCPARAMS, save_fig,
)

%matplotlib inline
matplotlib.rcParams.update(RCPARAMS)
matplotlib.rcParams['figure.dpi'] = 120   # screen resolution for notebook

## 1 — Configuration

In [ ]:
# ── Edit these paths ──────────────────────────────────────────────────────────
DATA_DIR  = Path("/Volumes/Seestar/MyWorks/postprod_m81/results/ES_UMa")

SIGMA     = 3.0    # sigma-clip threshold
BIN_MIN   = 5.0    # binning in minutes
PERIOD    = None   # period in days — set to float to enable phase plots
                   # e.g. 0.069 for ES UMa (~99 min orbital period)
T0        = None   # reference epoch JD for phase folding (None = first point)

SAVE_FIGS = True   # set False to skip saving to disk
FMT       = "both" # "pdf", "png", or "both"
# ─────────────────────────────────────────────────────────────────────────────

data = load_photometry(DATA_DIR / "photometry.csv")

print(f"Star       : {data['star_name']}")
print(f"Points     : {len(data['jd'])}")
print(f"Ensemble V : {data['ensemble_v']:.3f} mag" if data['ensemble_v'] else "Ensemble V : n/a")
print(f"JD range   : {data['jd'].min():.4f} – {data['jd'].max():.4f}")
print(f"Duration   : {(data['jd'].max()-data['jd'].min())*24:.2f} h")
if data['has_vapp']:
    print(f"V range    : {data['vapp'].min():.3f} – {data['vapp'].max():.3f} mag")
if data['fwhm'] is not None:
    print(f"FWHM range : {data['fwhm'].min():.2f} – {data['fwhm'].max():.2f} arcsec")

## 2 — Light curve

In [ ]:
fig = plot_light_curve(data, bin_min=BIN_MIN, sigma=SIGMA)
if SAVE_FIGS:
    paths = save_fig(fig, DATA_DIR, "light_curve_pub", FMT)
    for p in paths: print(f"Saved: {p}")
else:
    plt.show()

## 3 — Data quality diagnostics

In [ ]:
fig = plot_diagnostics(data, sigma=SIGMA)
if SAVE_FIGS:
    paths = save_fig(fig, DATA_DIR, "diagnostics_pub", FMT)
    for p in paths: print(f"Saved: {p}")
else:
    plt.show()

## 4 — Summary statistics

In [ ]:
mag  = data['vapp'] if data['has_vapp'] else data['vc']
err  = data['err']
mask = sigma_clip(mag, err, SIGMA)

mag_c = mag[mask]
err_c = err[mask]

print("─" * 45)
print(f"  Star               : {data['star_name']}")
print(f"  Quantity           : {'V_app' if data['has_vapp'] else 'V-C'}")
print(f"  Points (total)     : {len(mag)}")
print(f"  Points (clipped)   : {mask.sum()}  ({(~mask).sum()} rejected at {SIGMA}σ)")
print(f"  Mean               : {np.mean(mag_c):.4f} mag")
print(f"  Median             : {np.median(mag_c):.4f} mag")
print(f"  Std dev (rms)      : {np.std(mag_c):.4f} mag")
print(f"  Min / Max          : {mag_c.min():.4f} / {mag_c.max():.4f} mag")
print(f"  Amplitude (pk-pk)  : {mag_c.max()-mag_c.min():.4f} mag")
print(f"  Median σ_phot      : {np.median(err_c):.4f} mag")
print(f"  Mean SNR           : {1.0/np.median(err_c):.1f}")
print("─" * 45)

## 5 — Lomb-Scargle periodogram

In [ ]:
PMIN = 0.005   # minimum period to search [days]  → ~7 min
PMAX = 1.0     # maximum period to search [days]

fig = plot_periodogram(data, pmin=PMIN, pmax=PMAX, sigma=SIGMA)

# Find top-5 peaks
periods, power, best = lomb_scargle(
    data['jd'][mask], mag[mask], err[mask], pmin=PMIN, pmax=PMAX)
top5_idx = np.argsort(power)[::-1][:5]
print("Top 5 periods:")
for i, idx in enumerate(top5_idx, 1):
    p  = periods[idx]
    pw = power[idx]
    print(f"  {i}. P = {p:.5f} d = {p*1440:.2f} min   power = {pw:.4f}")

if SAVE_FIGS:
    paths = save_fig(fig, DATA_DIR, "periodogram_pub", FMT)
    for p in paths: print(f"Saved: {p}")
else:
    plt.show()

## 6 — Phase-folded light curve

Set `PERIOD` and optionally `T0` in the configuration cell above, then re-run.

In [ ]:
if PERIOD is None:
    # Use best period from periodogram if not manually set
    _, _, PERIOD_AUTO = lomb_scargle(data['jd'][mask], mag[mask], err[mask])
    print(f"Using auto-detected period: {PERIOD_AUTO:.5f} d = {PERIOD_AUTO*1440:.2f} min")
    period_use = PERIOD_AUTO
else:
    period_use = PERIOD
    print(f"Using manual period: {period_use:.5f} d = {period_use*1440:.2f} min")

fig = plot_phase_folded(data, period=period_use, t0=T0,
                        bin_min=max(0.5, BIN_MIN/5), sigma=SIGMA)
if SAVE_FIGS:
    paths = save_fig(fig, DATA_DIR, "phase_folded_pub", FMT)
    for p in paths: print(f"Saved: {p}")
else:
    plt.show()

## 7 — FWHM vs time (seeing quality)

Only available if `fwhm.csv` was produced by the pipeline.

In [ ]:
if data['fwhm'] is None:
    print("No fwhm.csv found — run the pipeline with FWHM extraction enabled.")
else:
    from scipy.ndimage import median_filter
    with plt.rc_context(RCPARAMS):
        fig, ax = plt.subplots(figsize=(7.2, 3.5))
        fwhm_jd  = data['fwhm_jd']
        fwhm_val = data['fwhm']
        off      = int(fwhm_jd.min())
        t        = fwhm_jd - off

        ax.scatter(t, fwhm_val, s=8, color='#27ae60', alpha=0.55,
                   linewidths=0, zorder=3, label='Per-frame FWHM')

        idx  = np.argsort(t)
        kern = min(21, max(3, len(fwhm_val)//10) | 1)
        ax.plot(t[idx], median_filter(fwhm_val[idx], size=kern),
                color='#1a5e2e', linewidth=1.5, zorder=4, label='Running median')

        ax.set_xlabel(f'JD−{off:,}  [d]')
        ax.set_ylabel('FWHM  [arcsec]')
        ax.set_ylim(bottom=0)
        ax.set_title(f"{data['star_name']} — Seeing (registration FWHM)",
                     fontweight='bold')
        ax.legend()
        import matplotlib.ticker as mticker
        ax.xaxis.set_minor_locator(mticker.AutoMinorLocator(5))
        ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(4))

        print(f"Median FWHM : {np.median(fwhm_val):.2f} arcsec")
        print(f"FWHM range  : {fwhm_val.min():.2f} – {fwhm_val.max():.2f} arcsec")

        if SAVE_FIGS:
            paths = save_fig(fig, DATA_DIR, "fwhm_pub", FMT)
            for p in paths: print(f"Saved: {p}")
        else:
            plt.show()

## 8 — Export for LaTeX

Copy-paste ready values for a paper table.

In [ ]:
from datetime import datetime, timezone
jd0  = data['jd'].min()
# Approximate civil date from JD
mjd  = jd0 - 2400000.5
unix = (mjd - 40587) * 86400
dt   = datetime.fromtimestamp(unix, tz=timezone.utc)

print("LaTeX table row (copy-paste):")
print()
print(f"%% Observation: {dt.strftime('%Y-%m-%d')}")
qty  = r"$V$" if data['has_vapp'] else r"$V-C$"
print(f"{data['star_name']} & {dt.strftime('%Y %b %d')} & {len(mag_c)} "
      f"& {qty} & ${np.median(mag_c):.3f}$ & ${np.std(mag_c):.3f}$ "
      f"& ${np.median(err_c):.3f}$ \\\\")